# PerioVision: panoramic per-tooth stage model (PDCNN grades)

Runtime: Runtime > Change runtime type > T4 GPU. Before running, upload `crops.npz` to /content (Files panel, top level). Training takes about 30-40 min for 40 epochs on a T4.

In [ ]:
import os, shutil
shutil.rmtree('/content/repo', ignore_errors=True)
!git clone -q https://github.com/VruddithShetty/PerioVision-AI.git /content/repo
!pip -q install opencv-python-headless scikit-learn

In [ ]:
import hashlib, os, zipfile
P = '/content/crops.npz'
assert os.path.exists(P), 'crops.npz is not in /content: re-upload it to the top level of the Files panel'
print('size MB', round(os.path.getsize(P)/1e6, 1))
assert zipfile.is_zipfile(P), 'upload is corrupt: delete it and upload again'
h = hashlib.sha256(open(P,'rb').read()).hexdigest()
print(h)
assert h == '1e5824e085149a79c4c5d7dff188affed5d51d788e491f5cbd84d2211a296c71', 'checksum differs: upload again'
os.makedirs('/content/data', exist_ok=True)
shutil.copy(P, '/content/data/crops.npz')
print('data ready')

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch runtime to T4')

In [ ]:
%cd /content/repo/backend
!python -m research.panoramic_stage train --data /content/data --out /content/model --epochs 40

In [ ]:
%cd /content/repo/backend
!python -m research.panoramic_stage evaluate --data /content/data --model /content/model/model.pt --out /content/model/report.json

In [ ]:
from google.colab import files
files.download('/content/model/model.pt')
files.download('/content/model/report.json')